In [2]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, types # Export to DataBase

# read XPT file
df_alcohol = pd.read_sas("../data/questionaire_data/ALQ_L.xpt", format="xport", encoding="utf-8")

# show top 5 rows 
df_alcohol.head()

,SEQN,ALQ111,ALQ121,ALQ130,ALQ142,ALQ270,ALQ280,ALQ151,ALQ170
0,130378.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,130379.0,1.0,2.000000e+00,3.0,5.397605e-79,NaN,NaN,2.0,NaN
2,130380.0,1.0,1.000000e+01,1.0,5.397605e-79,NaN,NaN,2.0,NaN
3,130386.0,1.0,4.000000e+00,2.0,1.000000e+01,5.397605e-79,10.0,2.0,5.397605e-79
4,130387.0,1.0,5.397605e-79,NaN,NaN,NaN,NaN,2.0,NaN


In [3]:
print("Alcohol Consumption Questionaire Data Info:")
df_alcohol.info() 

Alcohol Consumption Questionaire Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6337 entries, 0 to 6336
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   SEQN    6337 non-null   float64
 1   ALQ111  5481 non-null   float64
 2   ALQ121  4922 non-null   float64
 3   ALQ130  4069 non-null   float64
 4   ALQ142  4082 non-null   float64
 5   ALQ270  2366 non-null   float64
 6   ALQ280  2362 non-null   float64
 7   ALQ151  4901 non-null   float64
 8   ALQ170  2358 non-null   float64
dtypes: float64(9)
memory usage: 445.7 KB


In [5]:
# Select and rename essential columns
    # We'll keep SEQN for merging

alcohol_use_columns = {
    'SEQN': 'Participant_ID',
    'ALQ111': 'Ever_Had_Alcohol',            # Ever had a drink of any kind of alcohol
    'ALQ151': 'Ever_Binge_Every_Day',        # Ever have 4/5 or more drinks every day
    'ALQ121': 'Drinking_Frequency',          # Past 12 mos how often drank alc bev (frequency)
    'ALQ130': 'Avg_Drinks_Per_Drinking_Day', # Avg # alcoholic drinks/day/past 12 mos
    'ALQ142': 'Binge_Days',                  # days have 4/5 drinks/past 12 mos (binge drinking indicator)
    'ALQ270': 'Binge_episodes_2hr',          # times 4/5 drinks in 2hrs/past 12 mos (binge drinking)
    'ALQ280': 'Heavy_binge_Times'            # times 8+ drinks in 1 day/past 12 mos (heavy binge drinking)
}

df_alcohol = df_alcohol[list(alcohol_use_columns.keys())].copy() # list(...) transfers dicts into lists, which then can be worked in dataframe. 
df_alcohol.rename(columns=alcohol_use_columns, inplace=True) 

print("--- Selected and Renamed Alcohol Use Questionaire Data Info ---")
df_alcohol.info()

--- Selected and Renamed Alcohol Use Questionaire Data Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6337 entries, 0 to 6336
Data columns (total 8 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Participant_ID               6337 non-null   float64
 1   Ever_Had_Alcohol             5481 non-null   float64
 2   Ever_Binge_Every_Day         4901 non-null   float64
 3   Drinking_Frequency           4922 non-null   float64
 4   Avg_Drinks_Per_Drinking_Day  4069 non-null   float64
 5   Binge_Days                   4082 non-null   float64
 6   Binge_episodes_2hr           2366 non-null   float64
 7   Heavy_binge_Times            2362 non-null   float64
dtypes: float64(8)
memory usage: 396.2 KB


In [6]:
df_alcohol = df_alcohol.convert_dtypes() 
df_alcohol # 1 = ‘Yes’； 2 = ‘No’

,Participant_ID,Ever_Had_Alcohol,Ever_Binge_Every_Day,Drinking_Frequency,Avg_Drinks_Per_Drinking_Day,Binge_Days,Binge_episodes_2hr,Heavy_binge_Times
0,130378,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,130379,1,2,2.0,3,0.0,<NA>,<NA>
2,130380,1,2,10.0,1,0.0,<NA>,<NA>
3,130386,1,2,4.0,2,10.0,0.0,10.0
4,130387,1,2,0.0,<NA>,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...
6332,142305,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
6333,142307,1,2,10.0,1,0.0,<NA>,<NA>
6334,142308,1,2,8.0,2,0.0,<NA>,<NA>
6335,142309,2,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [7]:
df_alcohol['Drinking_Frequency'] = pd.to_numeric(df_alcohol['Drinking_Frequency'], errors='coerce').round().astype('Int64')
df_alcohol['Binge_Days'] = pd.to_numeric(df_alcohol['Binge_Days'], errors='coerce').round().astype('Int64')
df_alcohol['Binge_episodes_2hr'] = pd.to_numeric(df_alcohol['Binge_episodes_2hr'], errors='coerce').round().astype('Int64')
df_alcohol['Heavy_binge_Times'] = pd.to_numeric(df_alcohol['Heavy_binge_Times'], errors='coerce').round().astype('Int64')
df_alcohol


,Participant_ID,Ever_Had_Alcohol,Ever_Binge_Every_Day,Drinking_Frequency,Avg_Drinks_Per_Drinking_Day,Binge_Days,Binge_episodes_2hr,Heavy_binge_Times
0,130378,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,130379,1,2,2,3,0,<NA>,<NA>
2,130380,1,2,10,1,0,<NA>,<NA>
3,130386,1,2,4,2,10,0,10
4,130387,1,2,0,<NA>,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...
6332,142305,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
6333,142307,1,2,10,1,0,<NA>,<NA>
6334,142308,1,2,8,2,0,<NA>,<NA>
6335,142309,2,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [8]:
# export as csv
file_path = "../data/questionaire_data/cleaned_alcohol_use.csv" 

try:
    df_alcohol.to_csv(file_path, index=False, encoding='utf-8')
    print(f"DataFrame successfully saved to: {file_path}")
except Exception as e:
    print(f"Error saving DataFrame to CSV: {e}")

DataFrame successfully saved to: ../data/questionaire_data/cleaned_alcohol_use.csv


In [9]:
from dotenv import dotenv_values

config = dotenv_values()

# define variables for the login
pg_user = config['POSTGRES_USER']  # align the key label with your .env file !
pg_host = config['POSTGRES_HOST']
pg_port = config['POSTGRES_PORT']
pg_db = config['POSTGRES_DB']
pg_schema = config['POSTGRES_SCHEMA']
pg_pass = config['POSTGRES_PASS']

# Now building the URL with the values from the .env file
url = f'postgresql://{pg_user}:{pg_pass}@{pg_host}:{pg_port}/{pg_db}'

engine = create_engine(url, echo=False) 

In [ ]:
df_alcohol.to_sql(name = 'alcohol_use_questionaire_data', con=engine, schema='capstone_group_3',if_exists='replace',index=False)


337